# Fraud Compliance Agent Notebook 02 — Plaid Sandbox lifecycle probes

**Fraud Compliance Agent · Phase 0 · September 2026**  
**Status:** Completed source observation — lifecycle semantics pending ADR-003 review  
**CRISP-DM phases:** Business understanding → Data understanding → Data preparation → Evaluation → Deployment  
**Decision supported:** P0-03 / ADR-003 — source revision, correction, and replay semantics

---

## In plain English

This notebook checks how a **practice transaction changes over time**. A payment can be pending, completed, corrected, removed, or accidentally received twice; our app needs to know which of those changes the data source can show.

Think of tracking a delivery: its status can change after you first see it. The notebook records only safe counts and states from a Plaid Sandbox example. It does not connect production accounts, decide whether a payment is fraudulent, or assume that an unobserved behaviour can never happen.

Read the steps as: set up a safe practice item, observe an update, compare the before and after states, then create a small review record.

<a id="purpose"></a>
## Purpose

**Question:** How do added, modified, removed, pending, posted, duplicate, and missing-ID cases behave or fail in the selected Plaid Sandbox flow?

This notebook creates a fresh, Sandbox-only dynamic Transactions Item and records counts and states only. It never displays credentials, tokens, account IDs, transaction IDs, descriptions, or values.

<a id="contents"></a>
## Contents

1. [Purpose](#purpose)
2. [Prepare the isolated Sandbox Item](#prepare)
3. [Create and observe a safe update](#update)
4. [Evaluate lifecycle evidence](#evaluate)
5. [Produce a review artifact](#artifact)

### Non-goals

- This is not production ingestion, webhook handling, or a completed canonical contract.
- A bounded Sandbox observation never proves production behaviour.
- Unobserved states remain `indeterminate`, not impossible.


<a id="prepare"></a>
## Step 1 — Prepare the isolated Sandbox Item

The next cell loads the local configuration by name only, creates a fresh dynamic Sandbox Item, and waits for its initial Sync response. It retains raw provider material only in memory.


### Prepare the isolated practice-item helpers

The next cell loads the ignored local configuration and defines bounded, redacted Plaid Sandbox helpers. It does not create an item yet, and it never prints credential values or provider responses.


In [ ]:
from __future__ import annotations

import hashlib
import json
import os
import time
from datetime import UTC, datetime
from pathlib import Path
from typing import Any
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen

from dotenv import load_dotenv


def find_repository_root(start: Path) -> Path:
    """Find the repository root without reading data or environment values.

    Args:
        start: Current local notebook directory.

    Returns:
        Repository root containing the local API environment example.

    Raises:
        RuntimeError: If the notebook is outside this repository.

    Side effects:
        None.
    """
    for candidate in (start, *start.parents):
        if (candidate / "apps" / "api" / ".env.example").is_file():
            return candidate
    raise RuntimeError("Run this notebook from inside the fraud-compliance-agent repository.")


REPOSITORY_ROOT = find_repository_root(Path.cwd().resolve())
ENV_PATH = REPOSITORY_ROOT / "apps" / "api" / ".env"
REPORT_PATH = REPOSITORY_ROOT / "docs" / "proposals" / "plaid-lifecycle-probes.observation.json"

if not ENV_PATH.is_file():
    raise RuntimeError("Missing ignored apps/api/.env. Configure PLAID_CLIENT_ID, PLAID_SECRET, and PLAID_ENV locally.")

load_dotenv(ENV_PATH, override=False)
REQUIRED_NAMES = ("PLAID_CLIENT_ID", "PLAID_SECRET", "PLAID_ENV")
missing_names = [name for name in REQUIRED_NAMES if not os.getenv(name)]
if missing_names:
    raise RuntimeError("Missing required local configuration: " + ", ".join(missing_names))
if os.environ["PLAID_ENV"].strip().lower() != "sandbox":
    raise RuntimeError("Notebook 02 is restricted to PLAID_ENV=sandbox.")

PLAID_BASE_URL = "https://sandbox.plaid.com"
print("Local configuration present: " + ", ".join(REQUIRED_NAMES))
print("Probe mode: fresh Sandbox-only Item; no existing customer account is used.")


class PlaidProbeError(RuntimeError):
    def __init__(self, error_type: str, error_code: str):
        self.error_type = error_type
        self.error_code = error_code
        super().__init__(f"Plaid Sandbox probe failed: {error_type}/{error_code}.")


def plaid_post(path: str, payload: dict[str, Any]) -> dict[str, Any]:
    """Call a Plaid Sandbox endpoint and retain its raw response only in memory.

    Args:
        path: Relative Sandbox endpoint path.
        payload: Non-secret request fields; local credentials are attached in memory.

    Returns:
        Decoded response for the bounded lifecycle probe.

    Raises:
        PlaidProbeError: For a redacted provider error category.
        RuntimeError: For a network or malformed-response failure.

    Side effects:
        Sends a Sandbox request; never logs or writes raw provider data.
    """
    request = Request(
        f"{PLAID_BASE_URL}{path}",
        data=json.dumps({"client_id": os.environ["PLAID_CLIENT_ID"], "secret": os.environ["PLAID_SECRET"], **payload}).encode("utf-8"),
        headers={"Content-Type": "application/json"},
        method="POST",
    )
    try:
        with urlopen(request, timeout=30) as response:
            return json.loads(response.read().decode("utf-8"))
    except HTTPError as exc:
        try:
            payload = json.loads(exc.read().decode("utf-8"))
            error_type = payload.get("error_type", "unknown")
            error_code = payload.get("error_code", "unknown")
        except (UnicodeDecodeError, json.JSONDecodeError):
            error_type, error_code = "unknown", "unknown"
        raise PlaidProbeError(error_type, error_code) from None
    except (URLError, TimeoutError, json.JSONDecodeError) as exc:
        raise RuntimeError("Plaid Sandbox network or response failure.") from exc


def read_sync_pages(access_token: str, cursor: str | None) -> tuple[list[dict[str, Any]], str | None, int]:
    """Read one complete Sync update. Restart from its original cursor on mutation."""
    for restart_count in range(4):
        pages: list[dict[str, Any]] = []
        page_cursor = cursor
        try:
            while True:
                request: dict[str, Any] = {"access_token": access_token}
                if page_cursor:
                    request["cursor"] = page_cursor
                page = plaid_post("/transactions/sync", request)
                pages.append(page)
                if not page.get("has_more", False):
                    return pages, page.get("next_cursor"), restart_count
                page_cursor = page.get("next_cursor")
                if not page_cursor:
                    raise RuntimeError("Plaid Sync pagination response lacked next_cursor.")
        except PlaidProbeError as exc:
            if exc.error_code != "TRANSACTIONS_SYNC_MUTATION_DURING_PAGINATION":
                raise
    raise RuntimeError("Plaid Sync changed during pagination too often; retry later.")


def wait_for_initial_sync(access_token: str) -> tuple[list[dict[str, Any]], str | None, dict[str, Any]]:
    """Wait for a bounded initial Sync response from an isolated Sandbox Item.

    Args:
        access_token: In-memory Sandbox token that must not be logged or persisted.

    Returns:
        Sync response pages, final page cursor, and sanitised readiness metadata.

    Raises:
        RuntimeError: If a required pagination cursor is absent.

    Side effects:
        Polls Plaid Sandbox; raw pages remain only in memory.
    """
    started = time.monotonic()
    pages_seen: list[dict[str, Any]] = []
    cursor: str | None = None
    restart_count = 0
    status = "TRANSACTIONS_UPDATE_STATUS_UNKNOWN"
    while True:
        pages, cursor, restarts = read_sync_pages(access_token, cursor)
        pages_seen.extend(pages)
        restart_count += restarts
        status = pages[-1].get("transactions_update_status", status)
        added_count = sum(len(page.get("added", [])) for page in pages_seen)
        if status in {"INITIAL_UPDATE_COMPLETE", "HISTORICAL_UPDATE_COMPLETE"} and added_count:
            return pages_seen, cursor, {"status": status, "wait_seconds": round(time.monotonic() - started, 2), "page_count": len(pages_seen), "pagination_restart_count": restart_count, "timed_out": False}
        if time.monotonic() - started >= 45:
            return pages_seen, cursor, {"status": status, "wait_seconds": round(time.monotonic() - started, 2), "page_count": len(pages_seen), "pagination_restart_count": restart_count, "timed_out": True}
        time.sleep(2)



### Create the isolated practice item

The next cell creates a documented Plaid Sandbox test item, exchanges its short-lived token in memory, and waits for an initial cursor. It prints only sanitised readiness metadata; tokens, identifiers, and raw records must remain in memory.


In [ ]:
public_token = plaid_post(
    "/sandbox/public_token/create",
    {
        "institution_id": "ins_109508",
        "initial_products": ["transactions"],
        "options": {"override_username": "user_transactions_dynamic", "override_password": "pass_good"},
    },
)["public_token"]
access_token = plaid_post("/item/public_token/exchange", {"public_token": public_token})["access_token"]

initial_pages, baseline_cursor, readiness = wait_for_initial_sync(access_token)
if not baseline_cursor:
    raise RuntimeError("Sandbox initial Sync did not return a cursor; lifecycle probing cannot continue.")

print("Initial Sync status:", readiness["status"])
print("Initial Sync pages:", readiness["page_count"])
print("Initial Sync timed out:", readiness["timed_out"])
print("Raw records and tokens remain in memory only.")


<a id="update"></a>
## Step 2 — Create and observe a safe update

The next cell creates one isolated Sandbox transaction, reads the resulting Sync update, then replays the same saved cursor twice. It reduces all responses to counts and lifecycle relationships only.


In [ ]:
def summarise_pages(pages: list[dict[str, Any]], known_account_ids: set[str]) -> dict[str, Any]:
    """Reduce raw Sync pages to safe lifecycle counts and boolean observations.

    Args:
        pages: In-memory Plaid Sync pages that must not be printed or persisted.
        known_account_ids: In-memory account IDs used only for mismatch counting.

    Returns:
        Sanitised counts and state observations without identifiers or transaction values.

    Side effects:
        None.
    """
    added = [transaction for page in pages for transaction in page.get("added", [])]
    modified = [transaction for page in pages for transaction in page.get("modified", [])]
    removed = [transaction for page in pages for transaction in page.get("removed", [])]
    all_transactions = [*added, *modified]
    missing_id_count = sum(not transaction.get("transaction_id") for transaction in all_transactions)
    account_mismatch_count = sum(
        bool(transaction.get("account_id")) and transaction.get("account_id") not in known_account_ids
        for transaction in all_transactions
    )
    return {
        "added_count": len(added),
        "modified_count": len(modified),
        "removed_count": len(removed),
        "pending_count": sum(bool(transaction.get("pending")) for transaction in all_transactions),
        "pending_to_posted_link_count": sum(bool(transaction.get("pending_transaction_id")) for transaction in added),
        "missing_transaction_id_count": missing_id_count,
        "account_mismatch_count": account_mismatch_count,
        "page_count": len(pages),
    }


known_account_ids = {
    account.get("account_id")
    for page in initial_pages
    for account in page.get("accounts", [])
    if account.get("account_id")
}
baseline_shape = summarise_pages(initial_pages, known_account_ids)

# This deliberately creates a single, fresh Sandbox transaction. Its text and value
# are never printed or placed in the report; only the resulting Sync counts are kept.
today = datetime.now(UTC).date().isoformat()
plaid_post(
    "/sandbox/transactions/create",
    {
        "access_token": access_token,
        "transactions": [{
            "amount": 1.23,
            "date_posted": today,
            "date_transacted": today,
            "description": "FCA_NOTEBOOK_02_LIFECYCLE_PROBE",
            "iso_currency_code": "GBP",
        }],
    },
)

update_pages, update_cursor, update_restarts = read_sync_pages(access_token, baseline_cursor)
update_shape = summarise_pages(update_pages, known_account_ids)

# Re-read from the same saved cursor twice. We compare a sanitised page shape only,
# never transaction IDs or values, to test deterministic duplicate replay behaviour.
replay_one_pages, _, replay_one_restarts = read_sync_pages(access_token, baseline_cursor)
replay_two_pages, _, replay_two_restarts = read_sync_pages(access_token, baseline_cursor)
replay_one_shape = summarise_pages(replay_one_pages, known_account_ids)
replay_two_shape = summarise_pages(replay_two_pages, known_account_ids)

print("Custom Sandbox transaction creation attempted: true")
print("Update added/modified/removed counts:", {key: update_shape[key] for key in ("added_count", "modified_count", "removed_count")})
print("Same-cursor replay page shapes consistent:", replay_one_shape == replay_two_shape)
print("No raw transaction values, IDs, or account references were displayed.")


<a id="evaluate"></a>
## Step 3 — Evaluate lifecycle evidence

The next cell classifies each lifecycle state as observed, not observed, or indeterminate. It does not infer correction semantics from a missing observation.


In [ ]:
def probe_state(observed: bool, negative_state: str = "indeterminate") -> str:
    """Label a lifecycle observation without treating absence as impossibility.

    Args:
        observed: Whether the bounded Sandbox probe saw the state.
        negative_state: Safe label when it was not observed.

    Returns:
        ``observed`` or the supplied non-assertive state label.

    Side effects:
        None.
    """
    return "observed" if observed else negative_state

lifecycle_observations = {
    "added": probe_state(update_shape["added_count"] > 0),
    "modified": probe_state(update_shape["modified_count"] > 0),
    "removed": probe_state(update_shape["removed_count"] > 0),
    "pending": probe_state(baseline_shape["pending_count"] + update_shape["pending_count"] > 0),
    "pending_to_posted_link": probe_state(baseline_shape["pending_to_posted_link_count"] + update_shape["pending_to_posted_link_count"] > 0),
    "duplicate_replay": probe_state(replay_one_shape == replay_two_shape),
    "missing_transaction_id": "observed" if update_shape["missing_transaction_id_count"] else "not_observed",
    "account_mismatch": "observed" if update_shape["account_mismatch_count"] else "not_observed",
}

report = {
    "artifact": "plaid-lifecycle-probes",
    "status": "observation_ready",
    "notebook": "02-plaid-sandbox-lifecycle-probes.ipynb",
    "run_at": datetime.now(UTC).isoformat(),
    "git_revision": "unavailable" if not (REPOSITORY_ROOT / ".git").exists() else "uncommitted-or-unavailable",
    "source_system": "plaid_sandbox",
    "endpoint_summary": ["/transactions/sync", "/sandbox/transactions/create"],
    "initial_sync": readiness,
    "baseline_shape": baseline_shape,
    "update_shape": update_shape,
    "replay_shape_consistent": replay_one_shape == replay_two_shape,
    "pagination_restart_counts": {"update": update_restarts, "replay_one": replay_one_restarts, "replay_two": replay_two_restarts},
    "lifecycle_observations": lifecycle_observations,
    "limitations": [
        "An indeterminate state was not observed during this bounded Sandbox run; it is not evidence that the state cannot occur.",
        "The dynamic Sandbox source can change multiple records during the probe; update counts are lifecycle evidence, not a one-to-one attribution to the custom transaction.",
        "Sandbox behaviour does not establish production correction, freshness, or webhook behaviour.",
        "This report contains counts and states only; raw records, values, tokens, and identifiers were discarded from the notebook output.",
    ],
}
payload = json.dumps(report, sort_keys=True, indent=2)
report["report_sha256"] = hashlib.sha256(payload.encode("utf-8")).hexdigest()

print("Lifecycle observation states:", lifecycle_observations)
print("Observation is proposed evidence only; no canonical correction semantics are approved.")


<a id="artifact"></a>
## Step 4 — Produce a review artifact

The final cell writes a sanitised report to `docs/proposals/`. Inspect it and complete the matching experiment record before seeking an ADR-003 decision.


In [ ]:
REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)
REPORT_PATH.write_text(json.dumps(report, sort_keys=True, indent=2) + "\n", encoding="utf-8")
print("Sanitised lifecycle report written:", REPORT_PATH.relative_to(REPOSITORY_ROOT))
print("Report SHA-256:", report["report_sha256"])
print("Before committing, inspect this report and complete the matching experiment record.")


## Review checklist

- [ ] No cell output contains secrets, raw provider payloads, identifiers, or transaction values.
- [ ] Each lifecycle result is marked observed, not observed, or indeterminate.
- [ ] The experiment record links the report digest and documents limitations.
- [ ] No correction or replay runtime semantics are approved by this notebook alone.
- [ ] Recommendation remains **proposed**: use this observation in ADR-003 review without treating it as runtime approval.
